In [1]:
# Filtering and Selection
# Raw monitoring data contains everything — ok servers, warning servers, critical servers, all mixed together. Filtering is how you isolate what needs attention. Every alert, every anomaly flag, every incident report starts with a filter.

In [40]:
import pandas as pd

# All three datasets — load once, use everywhere
df_metrics = pd.read_csv(
    "server_metrics.csv",
    parse_dates=["timestamp"],
    dtype={"server_id": "category"},
    na_values=["N/A", "null", "—"]
)

df_tickets = pd.read_csv(
    "incidents.csv",
    parse_dates=["created_at", "resolved_at"]
)

df_logs = pd.read_csv(
    "app_logs.csv",
    parse_dates=["timestamp"]
)

print(f"df_metrics : {df_metrics.shape}")
print(f"df_tickets : {df_tickets.shape}")
print(f"df_logs    : {df_logs.shape}")

df_metrics : (500, 7)
df_tickets : (200, 7)
df_logs    : (300, 5)


In [2]:
# selecting columns
import pandas as pd
df_metrics = pd.read_csv("server_metrics.csv")
df_metrics
df_metrics["cpu_pct"]
print(df_metrics[[ "server_id", "cpu_pct", "status"]])
print(df_metrics.select_dtypes(include="number")) #only numbers
print(df_metrics.select_dtypes(include="object")) #only objects - letters

    server_id  cpu_pct   status
0      srv-01     49.6       ok
1      srv-02     32.3       ok
2      srv-03     21.6       ok
3      srv-04     34.5       ok
4      srv-05     68.3       ok
..        ...      ...      ...
495    srv-01     88.7  warning
496    srv-02     41.8       ok
497    srv-03     97.5       ok
498    srv-04     42.6       ok
499    srv-05     58.9       ok

[500 rows x 3 columns]
     cpu_pct  memory_pct  response_ms  disk_pct
0       49.6        94.6        891.8      68.9
1       32.3        33.9       1046.1      69.1
2       21.6        96.0       1007.3      43.8
3       34.5        50.7        653.5      58.1
4       68.3        39.5        386.0      53.8
..       ...         ...          ...       ...
495     88.7        38.9        959.1      38.1
496     41.8        89.6       1135.6      39.7
497     97.5        62.9       1043.1      68.3
498     42.6        43.8        926.1      55.1
499     58.9        69.3       1045.4      93.7

[500 rows x 4 c

In [3]:
# Boolean filtering 
df_metrics[df_metrics["cpu_pct"] > 80 ]

,timestamp,server_id,cpu_pct,memory_pct,response_ms,disk_pct,status
5,2026-01-01 01:00:00,srv-01,82.0,43.6,641.4,68.5,ok
7,2026-01-01 01:00:00,srv-03,83.9,50.7,162.3,74.5,ok
10,2026-01-01 02:00:00,srv-01,96.6,82.7,1130.4,88.2,ok
11,2026-01-01 02:00:00,srv-02,92.8,36.0,275.4,32.9,ok
14,2026-01-01 02:00:00,srv-05,81.0,43.5,56.4,83.0,ok
...,...,...,...,...,...,...,...
483,2026-01-05 00:00:00,srv-04,96.2,56.1,94.8,32.0,ok
485,2026-01-05 01:00:00,srv-01,91.1,37.9,1149.3,50.4,warning
492,2026-01-05 02:00:00,srv-03,98.5,89.9,766.9,67.0,ok
495,2026-01-05 03:00:00,srv-01,88.7,38.9,959.1,38.1,warning


In [4]:
df_metrics[df_metrics["status"] == "critical"]

,timestamp,server_id,cpu_pct,memory_pct,response_ms,disk_pct,status
6,2026-01-01 01:00:00,srv-02,68.0,41.6,124.8,91.7,critical
13,2026-01-01 02:00:00,srv-04,62.9,39.6,972.5,34.8,critical
27,2026-01-01 05:00:00,srv-03,45.5,65.3,858.5,53.6,critical
30,2026-01-01 06:00:00,srv-01,91.8,46.3,216.6,61.8,critical
64,2026-01-01 12:00:00,srv-05,63.4,78.6,809.2,48.2,critical
92,2026-01-01 18:00:00,srv-03,29.3,74.1,908.0,67.9,critical
93,2026-01-01 18:00:00,srv-04,49.6,49.4,1048.9,44.5,critical
99,2026-01-01 19:00:00,srv-05,47.9,69.7,139.4,93.3,critical
210,2026-01-02 18:00:00,srv-01,89.1,66.0,1129.9,81.9,critical
214,2026-01-02 18:00:00,srv-05,43.4,45.9,98.4,31.2,critical


In [5]:
df_metrics[df_metrics["status"] != "warning"]

,timestamp,server_id,cpu_pct,memory_pct,response_ms,disk_pct,status
0,2026-01-01 00:00:00,srv-01,49.6,94.6,891.8,68.9,ok
1,2026-01-01 00:00:00,srv-02,32.3,33.9,1046.1,69.1,ok
2,2026-01-01 00:00:00,srv-03,21.6,96.0,1007.3,43.8,ok
3,2026-01-01 00:00:00,srv-04,34.5,50.7,653.5,58.1,ok
4,2026-01-01 00:00:00,srv-05,68.3,39.5,386.0,53.8,ok
...,...,...,...,...,...,...,...
494,2026-01-05 02:00:00,srv-05,38.0,95.6,1095.9,76.9,ok
496,2026-01-05 03:00:00,srv-02,41.8,89.6,1135.6,39.7,ok
497,2026-01-05 03:00:00,srv-03,97.5,62.9,1043.1,68.3,ok
498,2026-01-05 03:00:00,srv-04,42.6,43.8,926.1,55.1,ok


In [6]:
# Multiple conditions - AND / OR ? NOT
df_metrics [
    (df_metrics["cpu_pct"] > 80) & 
    (df_metrics["memory_pct"] > 80)
    ]

,timestamp,server_id,cpu_pct,memory_pct,response_ms,disk_pct,status
10,2026-01-01 02:00:00,srv-01,96.6,82.7,1130.4,88.2,ok
23,2026-01-01 04:00:00,srv-04,88.8,84.6,264.6,88.0,ok
24,2026-01-01 04:00:00,srv-05,83.8,90.9,415.7,37.2,ok
48,2026-01-01 09:00:00,srv-04,94.3,94.9,1102.1,54.1,ok
54,2026-01-01 10:00:00,srv-05,83.9,85.1,1047.1,89.4,ok
79,2026-01-01 15:00:00,srv-05,93.5,88.4,543.3,78.8,warning
84,2026-01-01 16:00:00,srv-05,82.5,83.7,154.9,62.1,ok
89,2026-01-01 17:00:00,srv-05,94.8,97.0,916.4,54.5,ok
106,2026-01-01 21:00:00,srv-02,84.2,98.0,1196.1,66.1,warning
107,2026-01-01 21:00:00,srv-03,94.6,87.8,334.5,59.3,ok


In [7]:
df_metrics[
    (df_metrics["cpu_pct"] > 90) | (df_metrics["response_ms"] > 1000)
    ]

,timestamp,server_id,cpu_pct,memory_pct,response_ms,disk_pct,status
1,2026-01-01 00:00:00,srv-02,32.3,33.9,1046.1,69.1,ok
2,2026-01-01 00:00:00,srv-03,21.6,96.0,1007.3,43.8,ok
10,2026-01-01 02:00:00,srv-01,96.6,82.7,1130.4,88.2,ok
11,2026-01-01 02:00:00,srv-02,92.8,36.0,275.4,32.9,ok
12,2026-01-01 02:00:00,srv-03,50.7,48.5,1003.0,53.2,ok
...,...,...,...,...,...,...,...
492,2026-01-05 02:00:00,srv-03,98.5,89.9,766.9,67.0,ok
494,2026-01-05 02:00:00,srv-05,38.0,95.6,1095.9,76.9,ok
496,2026-01-05 03:00:00,srv-02,41.8,89.6,1135.6,39.7,ok
497,2026-01-05 03:00:00,srv-03,97.5,62.9,1043.1,68.3,ok


In [8]:
df_metrics[-(df_metrics["status"] == "ok")]

,timestamp,server_id,cpu_pct,memory_pct,response_ms,disk_pct,status
6,2026-01-01 01:00:00,srv-02,68.0,41.6,124.8,91.7,critical
13,2026-01-01 02:00:00,srv-04,62.9,39.6,972.5,34.8,critical
18,2026-01-01 03:00:00,srv-04,29.4,78.5,924.9,66.5,warning
20,2026-01-01 04:00:00,srv-01,22.5,73.3,411.5,63.1,warning
26,2026-01-01 05:00:00,srv-02,53.0,45.1,187.8,51.9,warning
...,...,...,...,...,...,...,...
475,2026-01-04 23:00:00,srv-01,41.8,50.8,957.3,59.0,warning
478,2026-01-04 23:00:00,srv-04,49.6,41.3,545.1,39.3,warning
485,2026-01-05 01:00:00,srv-01,91.1,37.9,1149.3,50.4,warning
486,2026-01-05 01:00:00,srv-02,67.6,86.2,1181.6,48.7,critical


In [9]:
# isin() - filter by a list of values
df_metrics[df_metrics["server_id"].isin(["srv-01", "srv-04"])] # only specific servers

df_tickets = pd.read_csv("incidents.csv")
df_tickets

,ticket_id,server_id,category,priority,created_at,resolved_at,resolved
0,INC0001,srv-04,Service Down,2,2026-03-24 02:00:00,NaN,False
1,INC0002,srv-01,CPU High,3,2026-03-19 15:00:00,2026-03-21 12:00:00,True
2,INC0003,srv-03,Service Down,2,2026-04-02 10:00:00,NaN,False
3,INC0004,srv-02,Service Down,1,2026-03-13 20:00:00,NaN,False
4,INC0005,srv-04,Disk Full,2,2026-03-18 09:00:00,2026-03-21 01:00:00,True
...,...,...,...,...,...,...,...
195,INC0196,srv-03,Memory Leak,1,2026-04-04 07:00:00,2026-04-04 17:00:00,True
196,INC0197,srv-02,CPU High,2,2026-03-03 05:00:00,2026-03-03 22:00:00,True
197,INC0198,srv-05,CPU High,2,2026-04-03 20:00:00,2026-04-06 04:00:00,True
198,INC0199,srv-02,Service Down,3,2026-02-10 11:00:00,NaN,False


In [10]:
df_tickets[df_tickets["priority"].isin([1,2])] # only p1 and p2 tickets

,ticket_id,server_id,category,priority,created_at,resolved_at,resolved
0,INC0001,srv-04,Service Down,2,2026-03-24 02:00:00,NaN,False
2,INC0003,srv-03,Service Down,2,2026-04-02 10:00:00,NaN,False
3,INC0004,srv-02,Service Down,1,2026-03-13 20:00:00,NaN,False
4,INC0005,srv-04,Disk Full,2,2026-03-18 09:00:00,2026-03-21 01:00:00,True
5,INC0006,srv-03,Network Lag,1,2026-01-29 23:00:00,NaN,False
...,...,...,...,...,...,...,...
194,INC0195,srv-03,Memory Leak,2,2026-03-20 07:00:00,2026-03-21 05:00:00,True
195,INC0196,srv-03,Memory Leak,1,2026-04-04 07:00:00,2026-04-04 17:00:00,True
196,INC0197,srv-02,CPU High,2,2026-03-03 05:00:00,2026-03-03 22:00:00,True
197,INC0198,srv-05,CPU High,2,2026-04-03 20:00:00,2026-04-06 04:00:00,True


In [11]:
df_tickets[~df_tickets["category"].isin(["Network Lag", "Unknown"])] # exclude specific categories

,ticket_id,server_id,category,priority,created_at,resolved_at,resolved
0,INC0001,srv-04,Service Down,2,2026-03-24 02:00:00,NaN,False
1,INC0002,srv-01,CPU High,3,2026-03-19 15:00:00,2026-03-21 12:00:00,True
2,INC0003,srv-03,Service Down,2,2026-04-02 10:00:00,NaN,False
3,INC0004,srv-02,Service Down,1,2026-03-13 20:00:00,NaN,False
4,INC0005,srv-04,Disk Full,2,2026-03-18 09:00:00,2026-03-21 01:00:00,True
...,...,...,...,...,...,...,...
195,INC0196,srv-03,Memory Leak,1,2026-04-04 07:00:00,2026-04-04 17:00:00,True
196,INC0197,srv-02,CPU High,2,2026-03-03 05:00:00,2026-03-03 22:00:00,True
197,INC0198,srv-05,CPU High,2,2026-04-03 20:00:00,2026-04-06 04:00:00,True
198,INC0199,srv-02,Service Down,3,2026-02-10 11:00:00,NaN,False


In [12]:
# label based selection

df_metrics.loc[:,["server_id", "cpu_pct", "status"]] # all rows and specif columns


,server_id,cpu_pct,status
0,srv-01,49.6,ok
1,srv-02,32.3,ok
2,srv-03,21.6,ok
3,srv-04,34.5,ok
4,srv-05,68.3,ok
...,...,...,...
495,srv-01,88.7,warning
496,srv-02,41.8,ok
497,srv-03,97.5,ok
498,srv-04,42.6,ok


In [13]:
# rows matching condition, specific columns
df_metrics.loc[
    df_metrics["cpu_pct"] > 80,
    ["server_id", "cpu_pct", "memory_pct", "status"]
    ]

,server_id,cpu_pct,memory_pct,status
5,srv-01,82.0,43.6,ok
7,srv-03,83.9,50.7,ok
10,srv-01,96.6,82.7,ok
11,srv-02,92.8,36.0,ok
14,srv-05,81.0,43.5,ok
...,...,...,...,...
483,srv-04,96.2,56.1,ok
485,srv-01,91.1,37.9,warning
492,srv-03,98.5,89.9,ok
495,srv-01,88.7,38.9,warning


In [14]:
df_metrics.set_index("server_id").loc["srv-04"] # set server_id as index - direct lookup

,timestamp,cpu_pct,memory_pct,response_ms,disk_pct,status
server_id,,,,,,
srv-04,2026-01-01 00:00:00,34.5,50.7,653.5,58.1,ok
srv-04,2026-01-01 01:00:00,29.6,63.7,89.5,89.1,ok
srv-04,2026-01-01 02:00:00,62.9,39.6,972.5,34.8,critical
srv-04,2026-01-01 03:00:00,29.4,78.5,924.9,66.5,warning
srv-04,2026-01-01 04:00:00,88.8,84.6,264.6,88.0,ok
...,...,...,...,...,...,...
srv-04,2026-01-04 23:00:00,49.6,41.3,545.1,39.3,warning
srv-04,2026-01-05 00:00:00,96.2,56.1,94.8,32.0,ok
srv-04,2026-01-05 01:00:00,63.7,66.8,926.2,84.2,ok


In [15]:
# iloc - position based selection
print(df_metrics.iloc[0])
print(df_metrics.iloc[:5])
print(df_metrics.iloc[10:20,:3])
print(df_metrics.iloc[-1])
print(df_metrics.iloc[::10])

timestamp      2026-01-01 00:00:00
server_id                   srv-01
cpu_pct                       49.6
memory_pct                    94.6
response_ms                  891.8
disk_pct                      68.9
status                          ok
Name: 0, dtype: object
             timestamp server_id  cpu_pct  memory_pct  response_ms  disk_pct  \
0  2026-01-01 00:00:00    srv-01     49.6        94.6        891.8      68.9   
1  2026-01-01 00:00:00    srv-02     32.3        33.9       1046.1      69.1   
2  2026-01-01 00:00:00    srv-03     21.6        96.0       1007.3      43.8   
3  2026-01-01 00:00:00    srv-04     34.5        50.7        653.5      58.1   
4  2026-01-01 00:00:00    srv-05     68.3        39.5        386.0      53.8   

  status  
0     ok  
1     ok  
2     ok  
3     ok  
4     ok  
              timestamp server_id  cpu_pct
10  2026-01-01 02:00:00    srv-01     96.6
11  2026-01-01 02:00:00    srv-02     92.8
12  2026-01-01 02:00:00    srv-03     50.7
13  2026-01-0

In [16]:
#  loc vs iloc — when to use which
           # loc                       iloc
# Basedon    Labels / conditions         Integer position
# Rows       Index label or boolean        0, 1, 2...
# Columns    Column name                   0, 1, 2...
# Use when   You know the name             You know the position
# AIOps use  Filter by server, status      Slice first N rows of log

In [17]:
# query()
df_metrics[
    (df_metrics["cpu_pct"] > 80) &    # boolean filetr
    (df_metrics["memory_pct"] > 80) &
    (df_metrics["status"] == "critical")
]

,timestamp,server_id,cpu_pct,memory_pct,response_ms,disk_pct,status


In [18]:
df_metrics.query("cpu_pct > 80 and memory_pct > 80 and status == 'critical'") # query reads like SQL WHERE clause

,timestamp,server_id,cpu_pct,memory_pct,response_ms,disk_pct,status


In [19]:
# between() - range filtering
df_metrics[df_metrics["cpu_pct"].between(70,90)]


,timestamp,server_id,cpu_pct,memory_pct,response_ms,disk_pct,status
5,2026-01-01 01:00:00,srv-01,82.0,43.6,641.4,68.5,ok
7,2026-01-01 01:00:00,srv-03,83.9,50.7,162.3,74.5,ok
9,2026-01-01 01:00:00,srv-05,72.3,51.2,648.1,65.5,ok
14,2026-01-01 02:00:00,srv-05,81.0,43.5,56.4,83.0,ok
15,2026-01-01 03:00:00,srv-01,77.6,82.4,135.2,53.3,ok
...,...,...,...,...,...,...,...
470,2026-01-04 22:00:00,srv-01,70.3,92.1,754.5,86.8,ok
474,2026-01-04 22:00:00,srv-05,71.5,87.5,846.4,57.9,ok
476,2026-01-04 23:00:00,srv-02,85.0,88.3,1104.1,58.0,ok
482,2026-01-05 00:00:00,srv-03,74.4,39.6,608.4,31.8,ok


In [20]:
df_metrics[df_metrics["response_ms"].between(800,1200)]

,timestamp,server_id,cpu_pct,memory_pct,response_ms,disk_pct,status
0,2026-01-01 00:00:00,srv-01,49.6,94.6,891.8,68.9,ok
1,2026-01-01 00:00:00,srv-02,32.3,33.9,1046.1,69.1,ok
2,2026-01-01 00:00:00,srv-03,21.6,96.0,1007.3,43.8,ok
10,2026-01-01 02:00:00,srv-01,96.6,82.7,1130.4,88.2,ok
12,2026-01-01 02:00:00,srv-03,50.7,48.5,1003.0,53.2,ok
...,...,...,...,...,...,...,...
495,2026-01-05 03:00:00,srv-01,88.7,38.9,959.1,38.1,warning
496,2026-01-05 03:00:00,srv-02,41.8,89.6,1135.6,39.7,ok
497,2026-01-05 03:00:00,srv-03,97.5,62.9,1043.1,68.3,ok
498,2026-01-05 03:00:00,srv-04,42.6,43.8,926.1,55.1,ok


In [21]:
# nlargets() and nsmallest() - top/bottom N
# top highest cpu reading across fleet
df_metrics.nlargest(10, "cpu_pct") [["server_id", "timestamp", "cpu_pct"]]

,server_id,timestamp,cpu_pct
343,srv-04,2026-01-03 20:00:00,98.6
492,srv-03,2026-01-05 02:00:00,98.5
95,srv-01,2026-01-01 19:00:00,98.4
78,srv-04,2026-01-01 15:00:00,98.2
141,srv-02,2026-01-02 04:00:00,98.2
162,srv-03,2026-01-02 08:00:00,97.8
135,srv-01,2026-01-02 03:00:00,97.6
497,srv-03,2026-01-05 03:00:00,97.5
366,srv-02,2026-01-04 01:00:00,97.4
433,srv-04,2026-01-04 14:00:00,97.3


In [22]:
# top 5 slowest response times
df_metrics.nlargest(5, "response_ms")[["server_id", "timestamp", "response_ms"]]

,server_id,timestamp,response_ms
169,srv-05,2026-01-02 09:00:00,1196.4
345,srv-01,2026-01-03 21:00:00,1196.2
106,srv-02,2026-01-01 21:00:00,1196.1
382,srv-03,2026-01-04 04:00:00,1195.8
255,srv-01,2026-01-03 03:00:00,1184.7


In [23]:
# 5 lowest disk usage servers
df_metrics.nsmallest(5, "disk_pct")[["server_id", "timestamp", "disk_pct"]]

,server_id,timestamp,disk_pct
41,srv-02,2026-01-01 08:00:00,30.3
233,srv-04,2026-01-02 22:00:00,30.4
296,srv-02,2026-01-03 11:00:00,30.4
25,srv-01,2026-01-01 05:00:00,30.5
256,srv-02,2026-01-03 03:00:00,30.6


In [24]:
# Filtering tickets 
# open p1 incidents only - needs immediate attention
open_p1 = df_tickets[
    (df_tickets["priority"] == 1) & 
    (df_tickets["resolved"] == False)
    ]
print(open_p1)

    ticket_id server_id      category  priority           created_at  \
3     INC0004    srv-02  Service Down         1  2026-03-13 20:00:00   
5     INC0006    srv-03   Network Lag         1  2026-01-29 23:00:00   
11    INC0012    srv-04     Disk Full         1  2026-04-09 17:00:00   
40    INC0041    srv-01      CPU High         1  2026-01-13 18:00:00   
53    INC0054    srv-01   Memory Leak         1  2026-03-19 07:00:00   
71    INC0072    srv-01   Network Lag         1  2026-03-21 17:00:00   
79    INC0080    srv-05  Service Down         1  2026-01-21 15:00:00   
84    INC0085    srv-03      CPU High         1  2026-02-16 07:00:00   
85    INC0086    srv-01      CPU High         1  2026-02-28 09:00:00   
98    INC0099    srv-05   Memory Leak         1  2026-01-09 22:00:00   
107   INC0108    srv-01   Network Lag         1  2026-02-15 02:00:00   
108   INC0109    srv-03  Service Down         1  2026-04-07 09:00:00   
109   INC0110    srv-04      CPU High         1  2026-03-17 13:0

In [25]:
# unresolved CPU or memory incidents
computer_issues = df_tickets[
    (df_tickets["category"].isin(["CPU High", "Memory Leak"])) &
    (df_tickets["resolved"] == False)
]
print(computer_issues)

    ticket_id server_id     category  priority           created_at  \
8     INC0009    srv-02     CPU High         2  2026-01-09 08:00:00   
19    INC0020    srv-02     CPU High         2  2026-03-17 09:00:00   
21    INC0022    srv-04  Memory Leak         3  2026-01-06 16:00:00   
25    INC0026    srv-01  Memory Leak         2  2026-02-08 16:00:00   
40    INC0041    srv-01     CPU High         1  2026-01-13 18:00:00   
41    INC0042    srv-05     CPU High         3  2026-02-15 02:00:00   
52    INC0053    srv-01  Memory Leak         3  2026-02-05 22:00:00   
53    INC0054    srv-01  Memory Leak         1  2026-03-19 07:00:00   
55    INC0056    srv-02     CPU High         3  2026-01-26 19:00:00   
61    INC0062    srv-01     CPU High         2  2026-03-02 21:00:00   
66    INC0067    srv-05     CPU High         3  2026-03-01 08:00:00   
84    INC0085    srv-03     CPU High         1  2026-02-16 07:00:00   
85    INC0086    srv-01     CPU High         1  2026-02-28 09:00:00   
92    

In [26]:
# High priority tickets on specific servers
critical_servers = df_tickets[
    (df_tickets["server_id"].isin(["srv-04", "srv-02"])) & (df_tickets["priority"] == 1)
    ]
print(critical_servers)

    ticket_id server_id      category  priority           created_at  \
3     INC0004    srv-02  Service Down         1  2026-03-13 20:00:00   
11    INC0012    srv-04     Disk Full         1  2026-04-09 17:00:00   
22    INC0023    srv-04   Network Lag         1  2026-02-04 02:00:00   
34    INC0035    srv-04     Disk Full         1  2026-03-17 15:00:00   
42    INC0043    srv-04   Memory Leak         1  2026-02-22 01:00:00   
54    INC0055    srv-02     Disk Full         1  2026-03-09 16:00:00   
59    INC0060    srv-02   Network Lag         1  2026-04-03 10:00:00   
69    INC0070    srv-02   Memory Leak         1  2026-04-01 19:00:00   
80    INC0081    srv-02     Disk Full         1  2026-01-12 07:00:00   
86    INC0087    srv-04   Memory Leak         1  2026-01-29 08:00:00   
89    INC0090    srv-02  Service Down         1  2026-01-04 05:00:00   
101   INC0102    srv-04      CPU High         1  2026-01-29 08:00:00   
105   INC0106    srv-02  Service Down         1  2026-04-02 11:0

In [27]:
# Task  — server_metrics
# Filter all rows where cpu_pct > 85. Show only server_id, timestamp, cpu_pct, status. Print row count and head().
import pandas as pd
filtered_df = df_metrics.loc[
    df_metrics["cpu_pct"] > 85,
    ["server_id", "timestamp", "cpu_pct", "status"]
    ]
print(filtered_df)
print("Row count:", len(filtered_df))
pct = len(filtered_df) / len(df_metrics) * 100
print(f" Fleet cpu > 85 % : {pct:.1f}% of all readings")
print("head:", filtered_df.head())

    server_id            timestamp  cpu_pct   status
10     srv-01  2026-01-01 02:00:00     96.6       ok
11     srv-02  2026-01-01 02:00:00     92.8       ok
16     srv-02  2026-01-01 03:00:00     88.2       ok
23     srv-04  2026-01-01 04:00:00     88.8       ok
28     srv-04  2026-01-01 05:00:00     96.0       ok
..        ...                  ...      ...      ...
483    srv-04  2026-01-05 00:00:00     96.2       ok
485    srv-01  2026-01-05 01:00:00     91.1  warning
492    srv-03  2026-01-05 02:00:00     98.5       ok
495    srv-01  2026-01-05 03:00:00     88.7  warning
497    srv-03  2026-01-05 03:00:00     97.5       ok

[93 rows x 4 columns]
Row count: 93
 Fleet cpu > 85 % : 18.6% of all readings
head:    server_id            timestamp  cpu_pct status
10    srv-01  2026-01-01 02:00:00     96.6     ok
11    srv-02  2026-01-01 02:00:00     92.8     ok
16    srv-02  2026-01-01 03:00:00     88.2     ok
23    srv-04  2026-01-01 04:00:00     88.8     ok
28    srv-04  2026-01-01 05:0

In [36]:
# Task — server_metrics
# Filter rows where memory_pct > 85 AND response_ms > 900. These are your most degraded servers. Print count and head().
import pandas as pd

df = pd.read_csv("server_metrics.csv")

filtered_df = df[(df["memory_pct"] > 85) & (df["response_ms"] > 900)]

print("Degraded server count:", len(filtered_df))
degraded_rate = len(filtered_df) / len(df_metrics) * 100
print(f"Degraded rate: {degraded_rate}%")
print(filtered_df.head)

Degraded server count: 31
Degraded rate: 6.2%
<bound method NDFrame.head of                timestamp server_id  cpu_pct  memory_pct  response_ms  \
2    2026-01-01 00:00:00    srv-03     21.6        96.0       1007.3   
25   2026-01-01 05:00:00    srv-01     53.7        85.6       1039.8   
38   2026-01-01 07:00:00    srv-04     27.4        91.0       1085.5   
48   2026-01-01 09:00:00    srv-04     94.3        94.9       1102.1   
54   2026-01-01 10:00:00    srv-05     83.9        85.1       1047.1   
73   2026-01-01 14:00:00    srv-04     50.5        95.4       1091.2   
82   2026-01-01 16:00:00    srv-03     45.2        94.6       1143.2   
89   2026-01-01 17:00:00    srv-05     94.8        97.0        916.4   
101  2026-01-01 20:00:00    srv-02     32.8        91.9        995.9   
106  2026-01-01 21:00:00    srv-02     84.2        98.0       1196.1   
143  2026-01-02 04:00:00    srv-04     93.4        86.5       1159.8   
161  2026-01-02 08:00:00    srv-02     78.2        93.5     

In [37]:
# Task — incidents
# Filter all unresolved P1 tickets. Print count. Then filter further — unresolved P1 tickets on srv-04 only.
import pandas as pd

df = pd.read_csv("incidents.csv")

p1_unresolved = df[(df["priority"] == "1") & (df["resolved"] == False)]

print("Unresolved P1 count:", len(p1_unresolved))

srv04_issues = p1_unresolved[p1_unresolved["server_id"] == "srv-04"]
print("Unresolved P1 on srv-04:", len(srv04_issues))
print(srv04_issues.head())

Unresolved P1 count: 0
Unresolved P1 on srv-04: 0
Empty DataFrame
Columns: [ticket_id, server_id, category, priority, created_at, resolved_at, resolved]
Index: []


In [30]:
# Task  — incidents
# Use isin() to filter tickets where category is either CPU High or Memory Leak. From that result filter further where resolved == False. Print value_counts() of server_id in the final result.
import pandas as pd
df = pd.read_csv("incidents.csv")

filtered = df[df["category"].isin(["CPU High", "Memory Leak"])]

final_df = filtered[filtered["resolved"] == False]

server_counts = final_df["server_id"].value_counts()

print(server_counts)

server_id
srv-01    10
srv-04     5
srv-05     5
srv-02     4
srv-03     4
Name: count, dtype: int64


In [31]:
# Task  — app_logs
# Filter all rows where log_level == "ERROR" or log_level == "CRITICAL". Print count. Then use isin() to do the same filter. Confirm both return identical row counts.
df = pd.read_csv("app_logs.csv")
error_or = df[(df["log_level"] == "ERROR") | (df["log_level"] == "CRITICAL")]
count_or = len(error_or)
print("count using OR: ", count_or)

errors_isin = df[df["log_level"].isin(["ERROR", "CRITICAL"])]
count_isin = len(errors_isin)
print("count using isin():", count_isin)
print("count match:", count_or == count_isin)

count using OR:  82
count using isin(): 82
count match: True


In [32]:
# Task — app_logs
# Use nlargest() to find top 10 rows by — wait, app_logs has no numeric column to rank on. So instead:

# Filter logs where error_code is not null
# From that result filter where log_level == "CRITICAL"
# Print value_counts() of server_id — which server has most critical errors with error codes?
import pandas as pd
df = pd.read_csv("app_logs.csv")

filtered = df[df["error_code"].notna()]

critical_errors = filtered[filtered["log_level"] == "CRITICAL"]

server_counts = critical_errors["server_id"].value_counts()

print(server_counts)

server_id
srv-05    9
srv-04    8
srv-02    5
srv-01    5
srv-03    4
Name: count, dtype: int64


In [39]:
# Task — cross-dataset
# Use query() on df_metrics to find rows where cpu_pct > 80 and status == "critical". Then use boolean filtering to find the same rows. Confirm both return identical row counts. Write a comment explaining when you would use query() over boolean filtering.
import pandas as pd

df_metrics = pd.read_csv("server_metrics.csv")
query_df = df_metrics.query('cpu_pct > 80 and status == "critical"')
count_query = len(query_df)

print("Count using query():", count_query)

#ANOTHER METHOD

bool_df = df_metrics[(df_metrics["cpu_pct"] > 80) & (df_metrics["status"] == "critical")]
count_bool = len(bool_df)

print("Count using boolean filtering:", count_bool)
print("Counts match:", count_query == count_bool)


#ANOTHER METHOD
# Boolean OR filter
import pandas as pd

df_logs = pd.read_csv("app_logs.csv", parse_dates=["timestamp"])
print(df_logs.shape)
print(df_logs.head())

error_bool = df_logs[
    (df_logs["log_level"] == "ERROR") |
    (df_logs["log_level"] == "CRITICAL")
]

# isin() filter — same result, cleaner syntax
error_isin = df_logs[df_logs["log_level"].isin(["ERROR", "CRITICAL"])]

# Verify both match
print(f"Boolean OR count : {len(error_bool)}")
print(f"isin() count     : {len(error_isin)}")
print(f"Counts match     : {len(error_bool) == len(error_isin)}")

# AIOps breakdown
print(error_isin["log_level"].value_counts())
print(error_isin["server_id"].value_counts())


Count using query(): 5
Count using boolean filtering: 5
Counts match: True
(300, 5)
            timestamp server_id log_level                   message error_code
0 2026-01-01 07:26:00    srv-04      INFO  Connection timeout error       E004
1 2026-01-01 03:46:00    srv-02      INFO       Database query slow       E003
2 2026-01-08 16:29:00    srv-05     ERROR    CPU threshold exceeded       E001
3 2026-01-04 12:34:00    srv-05   WARNING         Disk usage at 90%        NaN
4 2026-01-04 01:25:00    srv-04  CRITICAL     Authentication failed       E002
Boolean OR count : 82
isin() count     : 82
Counts match     : True
log_level
ERROR       44
CRITICAL    38
Name: count, dtype: int64
server_id
srv-05    20
srv-02    20
srv-04    15
srv-03    15
srv-01    12
Name: count, dtype: int64


In [34]:
# Use query() when:
# - You want cleaner, more readable expressions (especially with many conditions)
# - Writing SQL-like logic (easy for analysts / dashboards)
# - Working interactively (Jupyter notebooks, quick analysis)

# Use boolean filtering when:
# - Column names have spaces or special characters
# - You need Python variables inside conditions (query needs @var syntax)
# - You want more flexibility/debugging in complex pipelines